In [5]:
import cv2
import numpy as np  
import open3d as o3d
import open3d.visualization as vis
import time

In [6]:
distance_threshold = 0.01 # трешхолд для поиска плоскости пола RANSAC
ransac_iterations = 1000 # количество итераций поиска пола
grid_step = 0.01 # размер клетки в метрах при разбиении пола на сетку
max_dist = 0.4 # высота от пола, на которой точки считаются припятствованиями
morph_kernel_size = 1 + 2 * int(0.01 / grid_step)

In [7]:
def project_to_plane(points, plane_model):
    [a, b, c, d] = plane_model
    points = np.asarray(points)
    distances = a * points[:, 0] + b * points[:, 1] + c * points[:, 2] + d

    projected_points = points - np.outer(distances, [a, b, c])
    return projected_points


def switch_to_plane_coords(projected, plane_v1, plane_v2):
    dot_v1 = np.dot(projected, plane_v1)
    dot_v2 = np.dot(projected, plane_v2)

    plane_coords = np.column_stack((dot_v1, dot_v2))
    x_max, x_min = np.max(plane_coords[:, 0]), np.min(plane_coords[:, 0])
    y_max, y_min = np.max(plane_coords[:, 1]), np.min(plane_coords[:, 1])
    return plane_coords, x_max, x_min, y_max, y_min


def visualise(np_arr):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np_arr)
    vis.draw_geometries([pcd])


def visualise_vector_basis(pcd, normal, plane_v1, plane_v2):
    lines = []
    origin = (0,0,0)
    for vec, color in zip([normal, plane_v1, plane_v2], [[1,0,0], [0,1,0], [0,0,1]]):
        line = o3d.geometry.LineSet()
        line.points = o3d.utility.Vector3dVector([origin, origin + np.array(vec)])
        line.lines = o3d.utility.Vector2iVector([[0, 1]])
        line.colors = o3d.utility.Vector3dVector([color])
        lines.append(line)
    o3d.visualization.draw_geometries([pcd] + lines)


In [ ]:
# pcd = o3d.io.read_point_cloud("test_pcd.ply")
pcd_obj = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(pcd_obj.path)

start = time.time()
pcd = pcd.voxel_down_sample(voxel_size=grid_step)
print(f"Downsampled: {time.time() - start:.4f} секунд")
plane_model, inliers = pcd.segment_plane(distance_threshold = distance_threshold,
                                         num_iterations = ransac_iterations,
                                         ransac_n = 3)
[a, b, c, d] = plane_model
print(f"Поиск пола: {time.time() - start:.4f} секунд")

#смотрим че там у нас за плоскость такая
inlier_points = pcd.select_by_index(inliers)
outlier_points = pcd.select_by_index(inliers, invert = True)

#строим нормаль к поверхности   
outliers = np.asarray(outlier_points.points)
normal = np.array([a, b, c])
#меняем направление нормали в сторону большего количества точек
condition = a*outliers[:, 0] + b*outliers[:, 1] + c*outliers[:, 2] + d < 0
if len(outliers[condition]) > len(outliers) // 2:
    outliers = outliers[condition]
    normal = -normal
else:
    outliers = outliers[~condition]

# берем 2 перпендикулярных вектора в плоскости
v1_norm = np.linalg.norm([b, -a, 0])
plane_v1 = np.asarray([b, -a, 0]) / v1_norm
plane_v2 = np.linalg.cross(normal, plane_v1)

#проецируем точки пола на плоскость пола
projected = project_to_plane(inlier_points.points, plane_model=plane_model)

print(f"Проекция точек пола на плоскость: {time.time() - start:.4f} секунд")

#переходим от координат в пространстве в координаты 2-х векторов в плоскости
plane_coords, x_max, x_min, y_max, y_min = switch_to_plane_coords(projected, plane_v1, plane_v2)

#строим двумерный массив, соответствующий плоскости
grid = np.zeros((
    int((y_max - y_min) // grid_step + 1), 
    int((x_max - x_min) // grid_step + 1)
))
# Инвертируем Y т.к. в opencv 0 по оси Y это самый верх
x_indices = ((plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - plane_coords[:, 1]) // grid_step).astype(int) 
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

grid[y_indices[valid_mask], x_indices[valid_mask]] = 1

# смотрим че получилось
grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE, np.ones((morph_kernel_size, morph_kernel_size)))
grid = cv2.morphologyEx(grid, cv2.MORPH_OPEN, np.ones((morph_kernel_size, morph_kernel_size)))
floor_grid = np.copy(grid)

print(f"Составление сетки пола : {time.time() - start:.4f} секунд")

# берем только точки близкие к полу
condition = abs(a*outliers[:, 0] + b*outliers[:, 1] + c*outliers[:, 2] + d) <= max_dist
close_outliers = outliers[condition]

# проецируем на плоскость
projected_outliers = project_to_plane(close_outliers, plane_model=plane_model)

print(f"Проекция точек препятствия на пол: {time.time() - start:.4f} секунд")

# убираем из плоскости точки, где есть препятствия
outliers_plane_coords = switch_to_plane_coords(projected_outliers, plane_v1, plane_v2)[0]
x_indices = ((outliers_plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - outliers_plane_coords[:, 1]) // grid_step).astype(int)
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

grid[y_indices[valid_mask], x_indices[valid_mask]] = 0

print(f"Составление сетки пола с препятствиями (конец): {time.time() - start:.4f} секунд")

Downsampled: 0.0616 секунд
Поиск пола: 0.0881 секунд
Проекция точек пола на плоскость: 2.3710 секунд
Составление сетки пола : 2.3747 секунд
Проекция точек препятствия на пол: 2.3772 секунд
Составление сетки пола с препятствиями (конец): 2.3782 секунд


In [ ]:
# смотрим прогресс шаг за шагом
# Исходный pcd
vis.draw_geometries([pcd])

#выделенная плоскость
inlier_points.paint_uniform_color([0, 1, 0])
outlier_points.paint_uniform_color([1, 0, 0])
vis.draw_geometries([inlier_points, outlier_points])

#проекция точек пола на плоскость пола
visualise(projected)

#сетка пола
cv2.imshow('Floor, картина в цвете', (floor_grid * 255).astype(np.uint8))
cv2.waitKey(0)
cv2.destroyAllWindows()

# Точки, считающиеся прептствиями
visualise(close_outliers)

# Их проекция на плоскость пола
visualise(projected_outliers)

# Конецчный резальтат
grid = cv2.morphologyEx(grid, cv2.MORPH_OPEN, np.ones((morph_kernel_size, morph_kernel_size)))
grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE, np.ones((morph_kernel_size, morph_kernel_size)))
cv2.imshow('Floor with obstacles', (grid * 255).astype(np.uint8))
cv2.waitKey(0)
cv2.destroyAllWindows()